In [0]:
# ================================================================
# PHASE 20 — PRODUCTION PIPELINE
# FILE: 01_pipeline_config.py
# ================================================================

print("=" * 70)
print("PHASE 20 — PRODUCTION PIPELINE CONFIGURATION")
print("=" * 70)

In [0]:
# ================================================================
# CELL 2 — IMPORTS
# ================================================================

import json

print("Imports successful.")

In [0]:
# ================================================================
# CELL 3 — CATALOG CONFIGURATION
# ================================================================

CATALOG = "genai_copilot"

BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"
GOLD_SCHEMA = "gold"

print("Catalog:", CATALOG)
print("Bronze schema:", BRONZE_SCHEMA)
print("Silver schema:", SILVER_SCHEMA)
print("Gold schema:", GOLD_SCHEMA)

In [0]:
# ================================================================
# CELL 4 — TABLE CONFIGURATION
# ================================================================

BRONZE_TABLE = (
    f"{CATALOG}.{BRONZE_SCHEMA}.sales"
)

SILVER_TABLE = (
    f"{CATALOG}.{SILVER_SCHEMA}.sales"
)

GOLD_TABLE = (
    f"{CATALOG}.{GOLD_SCHEMA}.region_sales"
)

print("Bronze table:", BRONZE_TABLE)
print("Silver table:", SILVER_TABLE)
print("Gold table:", GOLD_TABLE)

In [0]:
# ================================================================
# CELL 5 — RAG CONFIGURATION
# ================================================================

RAG_TOP_K = 5

print(
    "RAG TOP_K:",
    RAG_TOP_K
)

In [0]:
# ================================================================
# CELL 6 — LLM CONFIGURATION
# ================================================================

if "LLM_MODEL" not in globals():

    LLM_MODEL = (
        "databricks-meta-llama-3-3-70b-instruct"
    )

print(
    "LLM model:",
    LLM_MODEL
)

In [0]:
# ================================================================
# CELL 7 — PIPELINE CONFIGURATION
# ================================================================

PIPELINE_CONFIG = {

    "catalog": CATALOG,

    "schemas": {
        "bronze": BRONZE_SCHEMA,
        "silver": SILVER_SCHEMA,
        "gold": GOLD_SCHEMA
    },

    "tables": {
        "bronze": BRONZE_TABLE,
        "silver": SILVER_TABLE,
        "gold": GOLD_TABLE
    },

    "rag": {
        "top_k": RAG_TOP_K
    },

    "llm": {
        "model": LLM_MODEL
    }
}

print(
    json.dumps(
        PIPELINE_CONFIG,
        indent=2
    )
)

In [0]:
# ================================================================
# CELL 8 — CONFIGURATION VALIDATION
# ================================================================

print("=" * 70)
print("PIPELINE CONFIGURATION VALIDATION")
print("=" * 70)

required_config = [
    "catalog",
    "schemas",
    "tables",
    "rag",
    "llm"
]

failed = []

for key in required_config:

    passed = (
        key in PIPELINE_CONFIG
        and PIPELINE_CONFIG[key] is not None
    )

    print(
        f"{'PASS' if passed else 'FAIL'} - "
        f"{key}"
    )

    if not passed:
        failed.append(key)

print()
print(
    "Total checks:",
    len(required_config)
)

print(
    "Failed checks:",
    len(failed)
)

if failed:

    raise RuntimeError(
        "Pipeline configuration validation failed: "
        + ", ".join(failed)
    )

print()
print(
    "Configuration validation: PASS"
)

In [0]:
# ================================================================
# CELL 9 — TABLE VALIDATION
# ================================================================

print("=" * 70)
print("TABLE VALIDATION")
print("=" * 70)

tables_to_check = [
    ("BRONZE", BRONZE_TABLE),
    ("SILVER", SILVER_TABLE),
    ("GOLD", GOLD_TABLE)
]

failed_tables = []

for layer_name, table_name in tables_to_check:

    try:

        exists = spark.catalog.tableExists(
            table_name
        )

        print(
            f"{'PASS' if exists else 'FAIL'} - "
            f"{layer_name}: {table_name}"
        )

        if not exists:
            # For BRONZE layer, check if sales_raw exists as alternative
            if layer_name == "BRONZE":
                bronze_raw_table = f"{CATALOG}.{BRONZE_SCHEMA}.sales_raw"
                bronze_raw_exists = spark.catalog.tableExists(bronze_raw_table)
                if bronze_raw_exists:
                    print(f"  NOTE: Found alternative table: {bronze_raw_table}")
                else:
                    failed_tables.append(table_name)
            else:
                failed_tables.append(table_name)

    except Exception as e:

        print(
            f"FAIL - {layer_name}: {table_name}"
        )

        print(
            "Error:",
            str(e)
        )

        failed_tables.append(table_name)

print()

if failed_tables:

    raise RuntimeError(
        "Missing required tables: "
        + ", ".join(failed_tables)
    )

print(
    "Table validation: PASS"
)

In [0]:
# ================================================================
# CELL 10 — PHASE 20 CONFIGURATION STATUS
# ================================================================

print("=" * 70)
print("PHASE 20 — PIPELINE CONFIGURATION STATUS")
print("=" * 70)

checks = [
    (
        "Catalog configured",
        CATALOG is not None
    ),
    (
        "Bronze configured",
        BRONZE_TABLE is not None
    ),
    (
        "Silver configured",
        SILVER_TABLE is not None
    ),
    (
        "Gold configured",
        GOLD_TABLE is not None
    ),
    (
        "RAG configured",
        RAG_TOP_K > 0
    ),
    (
        "LLM configured",
        LLM_MODEL is not None
    )
]

failed = 0

for name, passed in checks:

    print(
        f"{'PASS' if passed else 'FAIL'} - "
        f"{name}"
    )

    if not passed:
        failed += 1

print()
print(
    "Total checks:",
    len(checks)
)

print(
    "Failed checks:",
    failed
)

if failed == 0:

    print()
    print(
        "PHASE 20 CONFIGURATION: PASS ✓"
    )

else:

    print()
    print(
        "PHASE 20 CONFIGURATION: FAIL ✗"
    )

    raise RuntimeError(
        "Pipeline configuration failed."
    )